# Spam & Phishing Detection — Text Classification

## Objective

In this notebook, I develop and evaluate classical machine learning models for detecting spam messages and phishing emails.

I use TF-IDF representations with Logistic Regression and Linear Support Vector Machines, comparing their classification performance on fixed validation datasets.

The datasets were prepared in `01_data_exploration.ipynb`, where I removed exact and normalized duplicates, identified repeated message templates, and created group-aware train, validation, and test partitions.

My evaluation focuses on precision, recall, F1-score, and false positives rather than accuracy alone.

I keep the test sets untouched during model selection.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42

# I locate the project root without hardcoding machine-specific paths.
PROJECT_ROOT = Path.cwd().resolve().parent

SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "splits"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# I verify the expected project structure.
assert SPLIT_DIR.is_dir(), (
    "Dataset split directory not found. "
    "Run this notebook from the notebooks directory."
)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Split directory:", SPLIT_DIR)
print("Environment ready.")

Project root: C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector
Split directory: C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector\data\processed\splits
Environment ready.


In [8]:
# I load the fixed SMS partitions prepared in Notebook 01.
sms_train = pd.read_parquet(SPLIT_DIR / "sms_train.parquet")
sms_val = pd.read_parquet(SPLIT_DIR / "sms_val.parquet")

# I intentionally leave the test partition unopened during model selection.

print("SMS TRAINING DATA")
print("Shape:", sms_train.shape)
print("Spam proportion:", f"{sms_train['label'].mean():.2%}")

print("\nSMS VALIDATION DATA")
print("Shape:", sms_val.shape)
print("Spam proportion:", f"{sms_val['label'].mean():.2%}")

# I verify the expected labels and absence of missing text.
for name, df in [("Train", sms_train), ("Validation", sms_val)]:
    assert df["label"].isin([0, 1]).all(), f"Invalid labels in {name}"
    assert df["text"].notna().all(), f"Missing text in {name}"
    assert df["text"].str.strip().ne("").all(), f"Empty text in {name}"

print("\nDataset validation: PASSED")

SMS TRAINING DATA
Shape: (3665, 2)
Spam proportion: 12.31%

SMS VALIDATION DATA
Shape: (733, 2)
Spam proportion: 12.28%

Dataset validation: PASSED


## 1. Baseline Model — Majority Class

I begin with a majority-class baseline to establish the minimum performance reference for my spam classifier.

Since legitimate messages make up approximately 87.7% of the training data, a classifier that always predicts the majority class can achieve relatively high accuracy without detecting spam.

I evaluate this baseline using accuracy, precision, recall, F1-score, and a confusion matrix. My subsequent models should demonstrate meaningful improvements in detecting spam rather than simply reproducing the class imbalance.

In [11]:
# I separate message text from the target labels.
X_train = sms_train["text"]
y_train = sms_train["label"]

X_val = sms_val["text"]
y_val = sms_val["label"]

# I verify that the feature and label counts match.
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)

print("Training messages:", len(X_train))
print("Validation messages:", len(X_val))

Training messages: 3665
Validation messages: 733


In [13]:
# I use a majority-class classifier as my initial baseline.
baseline_model = DummyClassifier(
    strategy="most_frequent"
)

baseline_model.fit(X_train, y_train)

# I generate predictions for the validation set.
baseline_predictions = baseline_model.predict(X_val)

print("Baseline model trained successfully.")

Baseline model trained successfully.


In [15]:
# I calculate the baseline classification metrics.
baseline_results = {
    "Model": "Majority Class",
    "Accuracy": accuracy_score(y_val, baseline_predictions),
    "Precision": precision_score(
        y_val, baseline_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_val, baseline_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_val, baseline_predictions, zero_division=0
    )
}

print("MAJORITY-CLASS BASELINE RESULTS")

for metric, value in baseline_results.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, baseline_predictions, labels=[0, 1]))

MAJORITY-CLASS BASELINE RESULTS
Model: Majority Class
Accuracy: 0.8772
Precision: 0.0000
Recall: 0.0000
F1: 0.0000

CONFUSION MATRIX
[[643   0]
 [ 90   0]]


## 2. TF-IDF + Logistic Regression

I train my first text classification model using TF-IDF features and Logistic Regression.

TF-IDF converts message text into numerical features based on the importance of words and phrases across the training corpus.

I use unigrams and bigrams to capture both individual words and short phrases that may indicate spam.

I place the vectorizer and classifier inside a scikit-learn Pipeline to ensure that preprocessing is fitted only on training data and applied consistently during validation and future inference.

I initially use the default classification threshold and avoid class weighting so I can establish a straightforward baseline before experimenting with improvements.

In [18]:
# I build my first text classification pipeline.
logreg_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    )
])

# I fit the entire pipeline using training data only.
logreg_pipeline.fit(X_train, y_train)

print("TF-IDF + Logistic Regression trained successfully.")

print(
    "Number of TF-IDF features:",
    len(logreg_pipeline.named_steps["tfidf"].vocabulary_)
)

TF-IDF + Logistic Regression trained successfully.
Number of TF-IDF features: 9029


In [20]:
# I generate predictions using my trained pipeline.
logreg_predictions = logreg_pipeline.predict(X_val)

# I also retain the predicted probabilities for later analysis.
logreg_probabilities = logreg_pipeline.predict_proba(X_val)[:, 1]

print("Validation predictions generated successfully.")

print(
    "Predicted spam messages:",
    int(logreg_predictions.sum())
)

print(
    "Actual spam messages:",
    int(y_val.sum())
)

Validation predictions generated successfully.
Predicted spam messages: 61
Actual spam messages: 90


In [22]:
# I evaluate the first Logistic Regression model.
logreg_results = {
    "Model": "TF-IDF + Logistic Regression",
    "Accuracy": accuracy_score(y_val, logreg_predictions),
    "Precision": precision_score(
        y_val, logreg_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_val, logreg_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_val, logreg_predictions, zero_division=0
    )
}

print("LOGISTIC REGRESSION VALIDATION RESULTS")

for metric, value in logreg_results.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, logreg_predictions, labels=[0, 1]))

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_val,
        logreg_predictions,
        target_names=["Legitimate", "Spam"],
        zero_division=0
    )
)

LOGISTIC REGRESSION VALIDATION RESULTS
Model: TF-IDF + Logistic Regression
Accuracy: 0.9604
Precision: 1.0000
Recall: 0.6778
F1: 0.8079

CONFUSION MATRIX
[[643   0]
 [ 29  61]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.96      1.00      0.98       643
        Spam       1.00      0.68      0.81        90

    accuracy                           0.96       733
   macro avg       0.98      0.84      0.89       733
weighted avg       0.96      0.96      0.96       733



## 3. TF-IDF + Linear Support Vector Machine

I train a Linear Support Vector Machine to compare its spam detection performance with Logistic Regression.

Both models use the same TF-IDF configuration, allowing me to examine how the classification algorithm affects performance.

I begin with the default regularization strength and without class weighting.

My evaluation focuses on spam precision, recall, F1-score, and the number of legitimate messages incorrectly classified as spam.

In [25]:
# I create a Linear SVM pipeline using the same
# TF-IDF configuration as Logistic Regression.

svm_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LinearSVC(
            C=1.0,
            class_weight=None,
            random_state=RANDOM_STATE,
            max_iter=5000
        )
    )
])

# I fit the pipeline using training data only.
svm_pipeline.fit(X_train, y_train)

print("TF-IDF + Linear SVM trained successfully.")

print(
    "Number of TF-IDF features:",
    len(svm_pipeline.named_steps["tfidf"].vocabulary_)
)

TF-IDF + Linear SVM trained successfully.
Number of TF-IDF features: 9029


In [27]:
# I generate predictions for the validation partition.
svm_predictions = svm_pipeline.predict(X_val)

# I retain decision scores for possible later threshold analysis.
svm_decision_scores = svm_pipeline.decision_function(X_val)

print("Validation predictions generated successfully.")

print("Predicted spam:", int(svm_predictions.sum()))
print("Actual spam:", int(y_val.sum()))

Validation predictions generated successfully.
Predicted spam: 89
Actual spam: 90


In [29]:
# I evaluate Linear SVM using the same metrics
# applied to the previous models.

svm_results = {
    "Model": "TF-IDF + Linear SVM",
    "Accuracy": accuracy_score(y_val, svm_predictions),
    "Precision": precision_score(
        y_val, svm_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_val, svm_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_val, svm_predictions, zero_division=0
    )
}

print("LINEAR SVM VALIDATION RESULTS")

for metric, value in svm_results.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, svm_predictions, labels=[0, 1]))

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_val,
        svm_predictions,
        target_names=["Legitimate", "Spam"],
        zero_division=0
    )
)

LINEAR SVM VALIDATION RESULTS
Model: TF-IDF + Linear SVM
Accuracy: 0.9905
Precision: 0.9663
Recall: 0.9556
F1: 0.9609

CONFUSION MATRIX
[[640   3]
 [  4  86]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.99      1.00      0.99       643
        Spam       0.97      0.96      0.96        90

    accuracy                           0.99       733
   macro avg       0.98      0.98      0.98       733
weighted avg       0.99      0.99      0.99       733



## 4. Linear SVM — Error Analysis

I examine the validation messages misclassified by my Linear SVM model.

Although the model achieves strong validation performance, I want to understand the characteristics of its mistakes before deciding whether additional tuning is necessary.

I distinguish false positives from false negatives because they have different practical consequences. I also retain the original message text for inspection without modifying the validation dataset.

In [32]:
# I create a table containing the validation predictions.
sms_error_analysis = pd.DataFrame({
    "text": X_val.to_numpy(),
    "actual_label": y_val.to_numpy(),
    "predicted_label": svm_predictions,
    "decision_score": svm_decision_scores
})

# I retain only incorrectly classified messages.
sms_errors = sms_error_analysis[
    sms_error_analysis["actual_label"]
    != sms_error_analysis["predicted_label"]
].copy()

# I identify the type of classification error.
sms_errors["error_type"] = np.where(
    sms_errors["actual_label"] == 0,
    "False Positive",
    "False Negative"
)

print("LINEAR SVM ERROR ANALYSIS")
print("Total errors:", len(sms_errors))

print("\nError distribution:")
print(sms_errors["error_type"].value_counts())

display(
    sms_errors[
        [
            "error_type",
            "actual_label",
            "predicted_label",
            "decision_score",
            "text"
        ]
    ].sort_values("error_type")
)

LINEAR SVM ERROR ANALYSIS
Total errors: 7

Error distribution:
error_type
False Negative    4
False Positive    3
Name: count, dtype: int64


,error_type,actual_label,predicted_label,decision_score,text
191,False Negative,1,0,-0.150079,Bought one ringtone and now getting texts cost...
249,False Negative,1,0,-0.407684,Would you like to see my XXX pics they are so ...
448,False Negative,1,0,-1.123525,Sorry I missed your call let's talk when you h...
648,False Negative,1,0,-0.504806,"Hi this is Amy, we will be sending you a free ..."
65,False Positive,0,1,0.012843,"Hi hope u get this txt~journey hasnt been gd,n..."
163,False Positive,0,1,0.011034,We have sent JD for Customer Service cum Accou...
317,False Positive,0,1,0.219065,"Hi, Mobile no. &lt;#&gt; has added you in th..."


## 5. Character-Level TF-IDF Experiment

I investigate whether character-level TF-IDF features improve SMS spam detection.

My previous Linear SVM model achieved strong validation performance but missed four spam messages and incorrectly flagged three legitimate messages.

Character n-grams may help capture spelling variations, abbreviations, and unusual text patterns common in SMS messages.

I retain the same Linear SVM configuration and change only the TF-IDF representation, allowing me to evaluate whether this feature-engineering adjustment improves classification performance.

In [36]:
# I train a character-level TF-IDF + Linear SVM pipeline.
char_svm_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LinearSVC(
            C=1.0,
            class_weight=None,
            random_state=RANDOM_STATE,
            max_iter=5000
        )
    )
])

char_svm_pipeline.fit(X_train, y_train)

print("Character-level Linear SVM trained successfully.")

print(
    "Number of TF-IDF features:",
    len(char_svm_pipeline.named_steps["tfidf"].vocabulary_)
)

Character-level Linear SVM trained successfully.
Number of TF-IDF features: 57904


In [38]:
# I evaluate the character-level model on the same validation set.
char_svm_predictions = char_svm_pipeline.predict(X_val)

char_svm_results = {
    "Model": "Character TF-IDF + Linear SVM",
    "Accuracy": accuracy_score(y_val, char_svm_predictions),
    "Precision": precision_score(
        y_val, char_svm_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_val, char_svm_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_val, char_svm_predictions, zero_division=0
    )
}

print("CHARACTER-LEVEL LINEAR SVM RESULTS")

for metric, value in char_svm_results.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, char_svm_predictions, labels=[0, 1]))

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_val,
        char_svm_predictions,
        target_names=["Legitimate", "Spam"],
        zero_division=0
    )
)

CHARACTER-LEVEL LINEAR SVM RESULTS
Model: Character TF-IDF + Linear SVM
Accuracy: 0.9877
Precision: 0.9765
Recall: 0.9222
F1: 0.9486

CONFUSION MATRIX
[[641   2]
 [  7  83]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.99      1.00      0.99       643
        Spam       0.98      0.92      0.95        90

    accuracy                           0.99       733
   macro avg       0.98      0.96      0.97       733
weighted avg       0.99      0.99      0.99       733



## 6. SMS Model Selection

I compared Logistic Regression, word-level Linear SVM, and character-level Linear SVM using the same training and validation partitions.

The word-level Linear SVM achieved the highest validation F1-score of 96.09%, with 95.56% spam recall and 96.63% precision.

Although character-level features reduced the number of false positives slightly, they also increased missed spam detections.

I therefore select the word-level TF-IDF + Linear SVM as my SMS classification model.

I retain the chosen configuration and reserve the untouched test partition for final evaluation.

In [41]:
# I combine the SMS validation results into one comparison table.
sms_model_comparison = pd.DataFrame([
    baseline_results,
    logreg_results,
    svm_results,
    char_svm_results
])

sms_model_comparison = sms_model_comparison.sort_values(
    "F1",
    ascending=False
).reset_index(drop=True)

display(sms_model_comparison)

# I save the comparison for reproducibility.
sms_comparison_path = OUTPUTS_DIR / "sms_model_comparison.csv"

sms_model_comparison.to_csv(
    sms_comparison_path,
    index=False
)

print("\nSaved:", sms_comparison_path.name)

,Model,Accuracy,Precision,Recall,F1
0,TF-IDF + Linear SVM,0.990450,0.966292,0.955556,0.960894
1,Character TF-IDF + Linear SVM,0.987722,0.976471,0.922222,0.948571
2,TF-IDF + Logistic Regression,0.960437,1.000000,0.677778,0.807947
3,Majority Class,0.877217,0.000000,0.000000,0.000000



Saved: sms_model_comparison.csv


In [43]:
import joblib

# I retain the selected word-level SVM pipeline.
selected_sms_model = svm_pipeline

sms_model_path = MODELS_DIR / "sms_svm_pipeline.joblib"

joblib.dump(
    selected_sms_model,
    sms_model_path
)

# I verify that the saved pipeline can be reloaded.
reloaded_sms_model = joblib.load(sms_model_path)

assert np.array_equal(
    reloaded_sms_model.predict(X_val),
    svm_predictions
)

print("Selected SMS model saved successfully.")
print("Reload verification: PASSED")
print("Model file:", sms_model_path.name)

Selected SMS model saved successfully.
Reload verification: PASSED
Model file: sms_svm_pipeline.joblib


# Phishing Email Classification

I now extend my text classification experiments to phishing emails.

I use the cleaned and group-aware MeAJOR dataset prepared in Notebook 01.

My initial models operate on the combined email subject and body. I keep URL-specific features and image analysis outside this modeling stage so I can evaluate the contribution of text classification independently.

As with SMS detection, I begin by loading and validating the fixed training and validation partitions while keeping the test partition untouched.

In [46]:
# I load the phishing training and validation partitions.
phishing_train = pd.read_parquet(
    SPLIT_DIR / "phishing_train.parquet"
)

phishing_val = pd.read_parquet(
    SPLIT_DIR / "phishing_val.parquet"
)

print("PHISHING TRAINING DATA")
print("Shape:", phishing_train.shape)
print(
    "Phishing proportion:",
    f"{phishing_train['label'].mean():.2%}"
)

print("\nPHISHING VALIDATION DATA")
print("Shape:", phishing_val.shape)
print(
    "Phishing proportion:",
    f"{phishing_val['label'].mean():.2%}"
)

# I verify the modeling columns and labels.
for name, df in [
    ("Train", phishing_train),
    ("Validation", phishing_val)
]:
    assert {"text", "label"}.issubset(df.columns)
    assert df["label"].isin([0, 1]).all()
    assert df["text"].notna().all()
    assert df["text"].str.strip().ne("").all()

print("\nPhishing dataset validation: PASSED")

PHISHING TRAINING DATA
Shape: (73128, 6)
Phishing proportion: 44.60%

PHISHING VALIDATION DATA
Shape: (15606, 6)
Phishing proportion: 44.44%

Phishing dataset validation: PASSED


## 7. Phishing Classification — Majority-Class Baseline

I begin phishing email classification by establishing a majority-class baseline.

The dataset contains both legitimate and phishing emails, with legitimate messages representing approximately 55% of the observations.

I use a majority-class classifier to establish a reference for subsequent machine learning models.

I evaluate accuracy, phishing precision, recall, F1-score, and the confusion matrix to understand how a classifier performs without learning patterns from email text.

I keep the validation dataset separate from training and reserve the test partition for final evaluation.

In [49]:
# I separate email text from the classification labels.
X_phishing_train = phishing_train["text"]
y_phishing_train = phishing_train["label"]

X_phishing_val = phishing_val["text"]
y_phishing_val = phishing_val["label"]

# I verify that feature and label counts match.
assert len(X_phishing_train) == len(y_phishing_train)
assert len(X_phishing_val) == len(y_phishing_val)

print("Training emails:", len(X_phishing_train))
print("Validation emails:", len(X_phishing_val))

print("\nTraining label distribution:")
print(y_phishing_train.value_counts().sort_index())

print("\nValidation label distribution:")
print(y_phishing_val.value_counts().sort_index())

Training emails: 73128
Validation emails: 15606

Training label distribution:
label
0    40510
1    32618
Name: count, dtype: int64

Validation label distribution:
label
0    8671
1    6935
Name: count, dtype: int64


In [51]:
# I establish a baseline without learning from email content.
phishing_baseline = DummyClassifier(
    strategy="most_frequent"
)

phishing_baseline.fit(
    X_phishing_train,
    y_phishing_train
)

# I generate predictions on the validation partition.
phishing_baseline_predictions = phishing_baseline.predict(
    X_phishing_val
)

print("Phishing majority-class baseline trained successfully.")

Phishing majority-class baseline trained successfully.


In [53]:
# I calculate the phishing baseline metrics.
phishing_baseline_results = {
    "Model": "Majority Class",
    "Accuracy": accuracy_score(
        y_phishing_val,
        phishing_baseline_predictions
    ),
    "Precision": precision_score(
        y_phishing_val,
        phishing_baseline_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_phishing_val,
        phishing_baseline_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_phishing_val,
        phishing_baseline_predictions,
        zero_division=0
    )
}

print("PHISHING BASELINE RESULTS")

for metric, value in phishing_baseline_results.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

print("\nCONFUSION MATRIX")
print(
    confusion_matrix(
        y_phishing_val,
        phishing_baseline_predictions,
        labels=[0, 1]
    )
)

PHISHING BASELINE RESULTS
Model: Majority Class
Accuracy: 0.5556
Precision: 0.0000
Recall: 0.0000
F1: 0.0000

CONFUSION MATRIX
[[8671    0]
 [6935    0]]


## 8. Phishing Detection — TF-IDF + Logistic Regression

I train my first phishing text classifier using TF-IDF features extracted from the combined email subject and body.

I represent email text using word unigrams and bigrams to capture individual terms and short phrases associated with legitimate and phishing messages.

Because the phishing dataset is considerably larger than the SMS dataset, I limit the TF-IDF vocabulary to 100,000 features to control memory usage.

I use Logistic Regression as my initial classifier and evaluate its performance using phishing precision, recall, F1-score, accuracy, and a confusion matrix.

I fit the entire pipeline using the training partition only and evaluate it on the fixed validation partition.

In [56]:
import time

# I build a resource-conscious phishing text classification pipeline.
phishing_logreg_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            max_features=100_000,
            sublinear_tf=True,
            dtype=np.float32
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            class_weight=None,
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    )
])

# I measure the training time.
start_time = time.perf_counter()

phishing_logreg_pipeline.fit(
    X_phishing_train,
    y_phishing_train
)

training_time = time.perf_counter() - start_time

print("Phishing Logistic Regression trained successfully.")

print(
    "Number of TF-IDF features:",
    len(phishing_logreg_pipeline.named_steps["tfidf"].vocabulary_)
)

print(f"Training time: {training_time:.2f} seconds")

Phishing Logistic Regression trained successfully.
Number of TF-IDF features: 100000
Training time: 71.68 seconds


In [58]:
# I generate phishing predictions using the trained pipeline.
phishing_logreg_predictions = phishing_logreg_pipeline.predict(
    X_phishing_val
)

# I retain probability estimates for later threshold analysis.
phishing_logreg_probabilities = (
    phishing_logreg_pipeline.predict_proba(X_phishing_val)[:, 1]
)

print("Validation predictions generated successfully.")

print(
    "Predicted phishing emails:",
    int(phishing_logreg_predictions.sum())
)

print(
    "Actual phishing emails:",
    int(y_phishing_val.sum())
)

Validation predictions generated successfully.
Predicted phishing emails: 6932
Actual phishing emails: 6935


In [60]:
# I evaluate phishing classification performance.
phishing_logreg_results = {
    "Model": "TF-IDF + Logistic Regression",
    "Accuracy": accuracy_score(
        y_phishing_val,
        phishing_logreg_predictions
    ),
    "Precision": precision_score(
        y_phishing_val,
        phishing_logreg_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_phishing_val,
        phishing_logreg_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_phishing_val,
        phishing_logreg_predictions,
        zero_division=0
    )
}

print("PHISHING LOGISTIC REGRESSION RESULTS")

for metric, value in phishing_logreg_results.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

print("\nCONFUSION MATRIX")
print(
    confusion_matrix(
        y_phishing_val,
        phishing_logreg_predictions,
        labels=[0, 1]
    )
)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_phishing_val,
        phishing_logreg_predictions,
        target_names=["Legitimate", "Phishing"],
        zero_division=0
    )
)

PHISHING LOGISTIC REGRESSION RESULTS
Model: TF-IDF + Logistic Regression
Accuracy: 0.9812
Precision: 0.9791
Recall: 0.9787
F1: 0.9789

CONFUSION MATRIX
[[8526  145]
 [ 148 6787]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.98      0.98      0.98      8671
    Phishing       0.98      0.98      0.98      6935

    accuracy                           0.98     15606
   macro avg       0.98      0.98      0.98     15606
weighted avg       0.98      0.98      0.98     15606



## 9. Phishing Detection — Linear Support Vector Machine

I train a Linear Support Vector Machine using the same TF-IDF configuration as my Logistic Regression classifier.

This allows me to compare the classifiers under consistent feature-engineering conditions.

I evaluate the model using phishing precision, recall, F1-score, accuracy, and the confusion matrix.

I also record training time to assess whether the model is practical for a CPU-based detection system.

I retain the decision scores for later error analysis and possible threshold adjustments.

In [63]:
# I build a phishing Linear SVM pipeline.
phishing_svm_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            max_features=100_000,
            sublinear_tf=True,
            dtype=np.float32
        )
    ),
    (
        "classifier",
        LinearSVC(
            C=1.0,
            class_weight=None,
            random_state=RANDOM_STATE,
            max_iter=5000
        )
    )
])

# I measure the time needed to train the pipeline.
start_time = time.perf_counter()

phishing_svm_pipeline.fit(
    X_phishing_train,
    y_phishing_train
)

svm_training_time = time.perf_counter() - start_time

print("Phishing Linear SVM trained successfully.")

print(
    "Number of TF-IDF features:",
    len(phishing_svm_pipeline.named_steps["tfidf"].vocabulary_)
)

print(f"Training time: {svm_training_time:.2f} seconds")

Phishing Linear SVM trained successfully.
Number of TF-IDF features: 100000
Training time: 73.34 seconds


In [65]:
# I generate predictions using the trained SVM pipeline.
phishing_svm_predictions = phishing_svm_pipeline.predict(
    X_phishing_val
)

# I retain decision scores for subsequent analysis.
phishing_svm_scores = phishing_svm_pipeline.decision_function(
    X_phishing_val
)

print("Validation predictions generated successfully.")

print(
    "Predicted phishing emails:",
    int(phishing_svm_predictions.sum())
)

print(
    "Actual phishing emails:",
    int(y_phishing_val.sum())
)

Validation predictions generated successfully.
Predicted phishing emails: 6964
Actual phishing emails: 6935


In [67]:
# I evaluate the phishing Linear SVM classifier.
phishing_svm_results = {
    "Model": "TF-IDF + Linear SVM",
    "Accuracy": accuracy_score(
        y_phishing_val,
        phishing_svm_predictions
    ),
    "Precision": precision_score(
        y_phishing_val,
        phishing_svm_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_phishing_val,
        phishing_svm_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_phishing_val,
        phishing_svm_predictions,
        zero_division=0
    )
}

print("PHISHING LINEAR SVM VALIDATION RESULTS")

for metric, value in phishing_svm_results.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

print("\nCONFUSION MATRIX")
print(
    confusion_matrix(
        y_phishing_val,
        phishing_svm_predictions,
        labels=[0, 1]
    )
)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_phishing_val,
        phishing_svm_predictions,
        target_names=["Legitimate", "Phishing"],
        zero_division=0
    )
)

PHISHING LINEAR SVM VALIDATION RESULTS
Model: TF-IDF + Linear SVM
Accuracy: 0.9871
Precision: 0.9835
Recall: 0.9876
F1: 0.9855

CONFUSION MATRIX
[[8556  115]
 [  86 6849]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.99      0.99      0.99      8671
    Phishing       0.98      0.99      0.99      6935

    accuracy                           0.99     15606
   macro avg       0.99      0.99      0.99     15606
weighted avg       0.99      0.99      0.99     15606



## 10. Phishing Linear SVM — Error Analysis

I examine the validation errors produced by my phishing Linear SVM classifier.

Although the model achieves strong performance, I want to understand the characteristics of the incorrectly classified emails before finalizing my model selection.

I separate false positives from false negatives and examine the distribution of decision scores.

I also compare error rates across the original email sources to determine whether performance varies between the datasets combined in MeAJOR.

I do not modify the validation labels or use the test partition during this analysis.

In [72]:
# I create a validation-level prediction table.
phishing_error_analysis = phishing_val[
    ["source", "label", "subject", "text"]
].copy().reset_index(drop=True)

phishing_error_analysis["prediction"] = (
    phishing_svm_predictions
)

phishing_error_analysis["decision_score"] = (
    phishing_svm_scores
)

# I identify incorrectly classified emails.
phishing_errors = phishing_error_analysis[
    phishing_error_analysis["label"]
    != phishing_error_analysis["prediction"]
].copy()

phishing_errors["error_type"] = np.where(
    phishing_errors["label"] == 0,
    "False Positive",
    "False Negative"
)

print("PHISHING LINEAR SVM ERROR ANALYSIS")
print("Total validation emails:", len(phishing_error_analysis))
print("Total errors:", len(phishing_errors))

print("\nError distribution:")
print(phishing_errors["error_type"].value_counts())

print("\nDecision score statistics by error type:")
display(
    phishing_errors.groupby("error_type")[
        "decision_score"
    ].describe().round(4)
)

PHISHING LINEAR SVM ERROR ANALYSIS
Total validation emails: 15606
Total errors: 201

Error distribution:
error_type
False Positive    115
False Negative     86
Name: count, dtype: int64

Decision score statistics by error type:


,count,mean,std,min,25%,50%,75%,max
error_type,,,,,,,,
False Negative,86.0,-0.312,0.3346,-1.5981,-0.4214,-0.2114,-0.0963,-0.0022
False Positive,115.0,0.360,0.3297,0.0063,0.1227,0.2703,0.4760,1.6896


In [74]:
# I summarize model performance by original dataset source.
source_performance = (
    phishing_error_analysis
    .groupby("source")
    .apply(
        lambda group: pd.Series({
            "emails": len(group),
            "phishing_rate": group["label"].mean(),
            "accuracy": accuracy_score(
                group["label"],
                group["prediction"]
            ),
            "phishing_precision": precision_score(
                group["label"],
                group["prediction"],
                zero_division=0
            ),
            "phishing_recall": recall_score(
                group["label"],
                group["prediction"],
                zero_division=0
            ),
            "phishing_f1": f1_score(
                group["label"],
                group["prediction"],
                zero_division=0
            )
        }),
        include_groups=False
    )
    .reset_index()
)

display(source_performance.round(4))

,source,emails,phishing_rate,accuracy,phishing_precision,phishing_recall,phishing_f1
0,trec5,7000.0,0.4066,0.9849,0.9760,0.9870,0.9815
1,trec6,2207.0,0.2361,0.9841,0.9551,0.9789,0.9668
2,trec7,6399.0,0.5576,0.9906,0.9938,0.9893,0.9916


In [76]:
# I identify false positives with the strongest incorrect scores.
false_positives = (
    phishing_errors[
        phishing_errors["error_type"] == "False Positive"
    ]
    .sort_values("decision_score", ascending=False)
    .head(5)
)

# I identify false negatives with the strongest incorrect scores.
false_negatives = (
    phishing_errors[
        phishing_errors["error_type"] == "False Negative"
    ]
    .sort_values("decision_score", ascending=True)
    .head(5)
)

significant_errors = pd.concat(
    [false_positives, false_negatives],
    ignore_index=True
)

# I display limited text previews rather than full email bodies.
significant_errors["text_preview"] = (
    significant_errors["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 180)
)

display(
    significant_errors[
        [
            "error_type",
            "source",
            "decision_score",
            "subject",
            "text_preview"
        ]
    ]
)

,error_type,source,decision_score,subject,text_preview
0,False Positive,trec7,1.689597,[Reform] [FINANCIAL_INFO] [FINANCIAL_INFO],[Reform] [FINANCIAL_INFO] [FINANCIAL_INFO] [] ...
1,False Positive,trec5,1.571242,[USERNAME],[USERNAME] [USERNAME]<|EMAIL_SEPARATOR|>[USERN...
2,False Positive,trec5,1.504181,DRI Operating,"DRI Operating (.<|EMOJI|>,<|EMOJI|> <|EMOJI|>+..."
3,False Positive,trec5,1.014105,"[ORGANIZATION] Ship Notification, [REFERENCE_N...","[ORGANIZATION] Ship Notification, [REFERENCE_N..."
4,False Positive,trec5,0.983735,GREETINGS,"GREETINGS My dear [NAME], Sat Sri Akal. We are..."
5,False Negative,trec5,-1.598065,,test
6,False Negative,trec5,-1.518659,Sponsored tickets for Weather Trading Europe 2002,Sponsored tickets for Weather Trading Europe 2...
7,False Negative,trec7,-1.247033,precision depraved,precision depraved If it is the first paramete...
8,False Negative,trec5,-1.148954,New DOT Regulations for Welding Inspector Trai...,New DOT Regulations for Welding Inspector Trai...
9,False Negative,trec5,-1.118067,[ORGANIZATION] conference,[ORGANIZATION] conference The [ORGANIZATION] S...


## 11. Phishing Model Selection

I compared Logistic Regression and Linear SVM using identical TF-IDF configurations and the same fixed validation partition.

Linear SVM achieved the strongest performance, with a phishing F1-score of 98.55%, precision of 98.35%, and recall of 98.76%.

The classifier also produced fewer false positives and false negatives than Logistic Regression.

I examined prediction errors and performance across TREC5, TREC6, and TREC7. Although performance remained strong across the three sources, I observed differences in phishing recall and F1-score.

I also identified dataset preprocessing markers and examples of potentially ambiguous labels. These findings highlight limitations when applying the trained classifier to previously unseen, naturally occurring emails.

I therefore select TF-IDF + Linear SVM as my phishing text classifier while reserving the untouched test partition for final evaluation.

In [79]:
# I compare the phishing models using validation metrics.
phishing_model_comparison = pd.DataFrame([
    phishing_baseline_results,
    phishing_logreg_results,
    phishing_svm_results
])

phishing_model_comparison = (
    phishing_model_comparison
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

display(phishing_model_comparison)

# I save the comparison for project reporting.
comparison_path = OUTPUTS_DIR / "phishing_model_comparison.csv"

phishing_model_comparison.to_csv(
    comparison_path,
    index=False
)

print("Saved:", comparison_path.name)

,Model,Accuracy,Precision,Recall,F1
0,TF-IDF + Linear SVM,0.987120,0.983487,0.987599,0.985539
1,TF-IDF + Logistic Regression,0.981225,0.979083,0.978659,0.978871
2,Majority Class,0.555620,0.000000,0.000000,0.000000


Saved: phishing_model_comparison.csv


In [81]:
import joblib

# I select the strongest phishing classifier.
selected_phishing_model = phishing_svm_pipeline

phishing_model_path = (
    MODELS_DIR / "phishing_svm_pipeline.joblib"
)

# I save the complete TF-IDF + Linear SVM pipeline.
joblib.dump(
    selected_phishing_model,
    phishing_model_path
)

# I reload the model to verify serialization.
reloaded_phishing_model = joblib.load(
    phishing_model_path
)

# I confirm that its validation predictions remain unchanged.
assert np.array_equal(
    reloaded_phishing_model.predict(X_phishing_val),
    phishing_svm_predictions
)

print("Selected phishing model saved successfully.")
print("Reload verification: PASSED")
print("Model file:", phishing_model_path.name)

Selected phishing model saved successfully.
Reload verification: PASSED
Model file: phishing_svm_pipeline.joblib


## 12. Final Held-Out Test Evaluation

I evaluate my selected text classifiers on their previously untouched test partitions.

All model selection decisions were made using training and validation data. I now use the test datasets to estimate how well the selected configurations generalize to unseen examples from the prepared corpora.

I report accuracy, precision, recall, F1-score, and confusion matrices.

I do not use test results to tune hyperparameters, modify preprocessing, or select alternative models. Any weaknesses identified during testing will be documented as limitations or directions for future experiments.

In [84]:
# I load the previously untouched SMS test partition.
sms_test = pd.read_parquet(
    SPLIT_DIR / "sms_test.parquet"
)

# I verify the test dataset before evaluation.
assert len(sms_test) == 732
assert sms_test["label"].isin([0, 1]).all()
assert sms_test["text"].notna().all()

X_sms_test = sms_test["text"]
y_sms_test = sms_test["label"]

# I evaluate my previously selected SMS classifier.
sms_test_predictions = selected_sms_model.predict(
    X_sms_test
)

sms_test_results = {
    "Model": "TF-IDF + Linear SVM",
    "Dataset": "SMS Spam",
    "Accuracy": accuracy_score(
        y_sms_test, sms_test_predictions
    ),
    "Precision": precision_score(
        y_sms_test, sms_test_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_sms_test, sms_test_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_sms_test, sms_test_predictions,
        zero_division=0
    )
}

print("FINAL SMS TEST RESULTS")

for metric, value in sms_test_results.items():
    if metric in ["Model", "Dataset"]:
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

print("\nCONFUSION MATRIX")
print(
    confusion_matrix(
        y_sms_test,
        sms_test_predictions,
        labels=[0, 1]
    )
)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_sms_test,
        sms_test_predictions,
        target_names=["Legitimate", "Spam"],
        zero_division=0
    )
)

FINAL SMS TEST RESULTS
Model: TF-IDF + Linear SVM
Dataset: SMS Spam
Accuracy: 0.9809
Precision: 0.9872
Recall: 0.8556
F1: 0.9167

CONFUSION MATRIX
[[641   1]
 [ 13  77]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.98      1.00      0.99       642
        Spam       0.99      0.86      0.92        90

    accuracy                           0.98       732
   macro avg       0.98      0.93      0.95       732
weighted avg       0.98      0.98      0.98       732



In [86]:
# I compare validation performance against held-out test performance.
sms_final_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Validation": [
        svm_results["Accuracy"],
        svm_results["Precision"],
        svm_results["Recall"],
        svm_results["F1"]
    ],
    "Test": [
        sms_test_results["Accuracy"],
        sms_test_results["Precision"],
        sms_test_results["Recall"],
        sms_test_results["F1"]
    ]
})

sms_final_comparison["Difference"] = (
    sms_final_comparison["Test"]
    - sms_final_comparison["Validation"]
)

display(sms_final_comparison.round(4))

,Metric,Validation,Test,Difference
0,Accuracy,0.9905,0.9809,-0.0096
1,Precision,0.9663,0.9872,0.0209
2,Recall,0.9556,0.8556,-0.1000
3,F1,0.9609,0.9167,-0.0442


## 13. Final Phishing Test Evaluation

I evaluate my selected TF-IDF + Linear SVM phishing classifier using the previously untouched test partition.

The model was selected based on validation performance, where it achieved an F1-score of 98.55%.

I now measure its held-out accuracy, precision, recall, and F1-score to assess generalization within the prepared email corpus.

I compare validation and test performance without using the test results to modify the selected model.

In [89]:
# I load the previously untouched phishing test partition.
phishing_test = pd.read_parquet(
    SPLIT_DIR / "phishing_test.parquet"
)

# I validate the test dataset.
assert len(phishing_test) == 15617
assert phishing_test["label"].isin([0, 1]).all()
assert phishing_test["text"].notna().all()
assert phishing_test["text"].str.strip().ne("").all()

X_phishing_test = phishing_test["text"]
y_phishing_test = phishing_test["label"]

# I generate predictions using my previously selected model.
phishing_test_predictions = selected_phishing_model.predict(
    X_phishing_test
)

# I calculate the final held-out metrics.
phishing_test_results = {
    "Model": "TF-IDF + Linear SVM",
    "Dataset": "Phishing Emails",
    "Accuracy": accuracy_score(
        y_phishing_test,
        phishing_test_predictions
    ),
    "Precision": precision_score(
        y_phishing_test,
        phishing_test_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_phishing_test,
        phishing_test_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_phishing_test,
        phishing_test_predictions,
        zero_division=0
    )
}

print("FINAL PHISHING TEST RESULTS")

for metric, value in phishing_test_results.items():
    if metric in ["Model", "Dataset"]:
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

print("\nCONFUSION MATRIX")
print(
    confusion_matrix(
        y_phishing_test,
        phishing_test_predictions,
        labels=[0, 1]
    )
)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_phishing_test,
        phishing_test_predictions,
        target_names=["Legitimate", "Phishing"],
        zero_division=0
    )
)

FINAL PHISHING TEST RESULTS
Model: TF-IDF + Linear SVM
Dataset: Phishing Emails
Accuracy: 0.9883
Precision: 0.9861
Recall: 0.9878
F1: 0.9869

CONFUSION MATRIX
[[8564   97]
 [  85 6871]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.99      0.99      0.99      8661
    Phishing       0.99      0.99      0.99      6956

    accuracy                           0.99     15617
   macro avg       0.99      0.99      0.99     15617
weighted avg       0.99      0.99      0.99     15617



In [91]:
# I compare phishing validation and test results.
phishing_final_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Validation": [
        phishing_svm_results["Accuracy"],
        phishing_svm_results["Precision"],
        phishing_svm_results["Recall"],
        phishing_svm_results["F1"]
    ],
    "Test": [
        phishing_test_results["Accuracy"],
        phishing_test_results["Precision"],
        phishing_test_results["Recall"],
        phishing_test_results["F1"]
    ]
})

phishing_final_comparison["Difference"] = (
    phishing_final_comparison["Test"]
    - phishing_final_comparison["Validation"]
)

display(phishing_final_comparison.round(4))

,Metric,Validation,Test,Difference
0,Accuracy,0.9871,0.9883,0.0012
1,Precision,0.9835,0.9861,0.0026
2,Recall,0.9876,0.9878,0.0002
3,F1,0.9855,0.9869,0.0014


## 14. Final Model Evaluation Summary

I completed the development and held-out evaluation of two text classification models: one for SMS spam detection and another for phishing email detection.

For both tasks, I selected TF-IDF with Linear SVM based on validation performance.

My SMS classifier achieved 98.09% test accuracy and a 91.67% spam F1-score. Although precision remained high, recall decreased from 95.56% during validation to 85.56% during testing, indicating a limitation in detecting unfamiliar spam messages.

My phishing classifier achieved 98.83% test accuracy and a 98.69% phishing F1-score. Its performance remained consistent between validation and testing.

These results demonstrate that classical machine learning can provide strong text classification performance using CPU-compatible methods.

However, the evaluation uses historical datasets and does not establish equivalent performance on current phishing campaigns. I retain these limitations when interpreting the results.

In [94]:
# I consolidate the final held-out results for both classifiers.
final_results = pd.DataFrame([
    sms_test_results,
    phishing_test_results
])

# I organize the columns for reporting.
final_results = final_results[
    [
        "Dataset",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
]

print("FINAL HELD-OUT MODEL EVALUATION")
display(final_results.round(4))

# I save the final results for reporting and documentation.
final_results_path = (
    OUTPUTS_DIR / "final_text_model_results.csv"
)

final_results.to_csv(
    final_results_path,
    index=False
)

print("\nSaved:", final_results_path.name)

FINAL HELD-OUT MODEL EVALUATION


,Dataset,Model,Accuracy,Precision,Recall,F1
0,SMS Spam,TF-IDF + Linear SVM,0.9809,0.9872,0.8556,0.9167
1,Phishing Emails,TF-IDF + Linear SVM,0.9883,0.9861,0.9878,0.9869



Saved: final_text_model_results.csv


In [96]:
# I preserve confusion matrices from the final test evaluation.
confusion_results = {
    "sms": confusion_matrix(
        y_sms_test,
        sms_test_predictions,
        labels=[0, 1]
    ).tolist(),

    "phishing": confusion_matrix(
        y_phishing_test,
        phishing_test_predictions,
        labels=[0, 1]
    ).tolist()
}

import json

confusion_path = (
    OUTPUTS_DIR / "final_text_confusion_matrices.json"
)

with confusion_path.open("w", encoding="utf-8") as file:
    json.dump(confusion_results, file, indent=2)

print("Saved:", confusion_path.name)

Saved: final_text_confusion_matrices.json
